# CardioIA - Fase 6: Sistema Preditivo Multiagente para Eventos Cardíacos

Este notebook contém a implementação do modelo preditivo de pico de risco cardíaco (Parte 1), incluindo geração de dados, treinamento, avaliação e simulação de previsão para um novo paciente.

**Nota:** A implementação completa do sistema multiagente com OpenAI Agents SDK está disponível no repositório GitHub no arquivo `cardioia_agents.py`.

## 1. Instalação de Dependências
Primeiro, vamos instalar as bibliotecas necessárias para o projeto.

In [ ]:
!pip install scikit-learn joblib pandas matplotlib seaborn

## 2. Geração da Base de Dados Sintética e Treinamento do Modelo Preditivo

Nesta seção, geraremos uma base de dados sintética para simular dados clínicos de pacientes e, em seguida, treinaremos um modelo de Machine Learning para prever picos de risco cardíaco.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib

# 1. Geração da Base de Dados Sintética
def generate_synthetic_data(n_samples=1000):
    np.random.seed(42)
    data = {
        'idade': np.random.randint(20, 90, n_samples),
        'frequencia_cardiaca': np.random.randint(50, 120, n_samples),
        'spo2': np.random.randint(85, 100, n_samples),
        'carga_sistema': np.random.uniform(0.1, 1.0, n_samples),
        'disponibilidade_recursos': np.random.uniform(0.1, 1.0, n_samples),
        'historico_cardiaco': np.random.choice([0, 1], n_samples, p=[0.7, 0.3])
    }
    
    df = pd.DataFrame(data)
    
    # Lógica para definir o pico_risco (variável alvo)
    # Risco aumenta com idade, frequência cardíaca alta, spo2 baixo e carga do sistema alta
    score = (
        (df['idade'] / 90) * 0.2 +
        (df['frequencia_cardiaca'] / 120) * 0.3 +
        ((100 - df['spo2']) / 15) * 0.3 +
        (df['carga_sistema']) * 0.1 +
        (df['historico_cardiaco']) * 0.1
    )
    
    df['pico_risco'] = (score > 0.6).astype(int)
    return df

# 2. Treinamento do Modelo
def train_model(df):
    X = df.drop('pico_risco', axis=1)
    y = df['pico_risco']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'confusion_matrix': confusion_matrix(y_test, y_pred).tolist(),
        'report': classification_report(y_test, y_pred, output_dict=True)
    }
    
    return model, metrics, X_test, y_test, y_pred

df = generate_synthetic_data()
df.to_csv('base_cardioia.csv', index=False)
print("Base de dados gerada e salva em 'base_cardioia.csv'.")
print(f"Total de amostras: {len(df)}")
print(f"Distribuição de pico_risco:\n{df['pico_risco'].value_counts()}")

model, metrics, X_test, y_test, y_pred = train_model(df)
joblib.dump(model, 'modelo_cardioia.pkl')
print("\nModelo treinado e salvo em 'modelo_cardioia.pkl'.")

print(f"\nAcurácia: {metrics['accuracy']:.4f}")
print("Matriz de Confusão:")
print(np.array(metrics['confusion_matrix']))

## 3. Avaliação do Modelo e Simulação de Previsão

Aqui, avaliaremos o desempenho do modelo utilizando a matriz de confusão e simularemos a previsão para um novo paciente.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Avaliação Visual
def plot_evaluation(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Sem Pico', 'Pico de Risco'],
                yticklabels=['Sem Pico', 'Pico de Risco'])
    plt.title('Matriz de Confusão - CardioIA')
    plt.ylabel('Real')
    plt.xlabel('Predito')
    plt.tight_layout()
    plt.show()

# 2. Simulação de Novo Paciente
def simulate_new_patient(model):
    # Simulação de um paciente com alto risco
    new_patient = pd.DataFrame([{
        'idade': 75,
        'frequencia_cardiaca': 110,
        'spo2': 88,
        'carga_sistema': 0.8,
        'disponibilidade_recursos': 0.4,
        'historico_cardiaco': 1
    }])
    
    prob = model.predict_proba(new_patient)[0][1]
    pred = model.predict(new_patient)[0]
    
    print("\n--- Simulação de Novo Paciente ---")
    print(new_patient.iloc[0])
    print(f"\nProbabilidade de Pico de Risco: {prob:.2%}")
    print(f"Classificação: {'Alto Risco' if pred == 1 else 'Baixo Risco'}")
    
    return prob, pred

plot_evaluation(y_test, y_pred)
simulate_new_patient(model)

## 4. Demonstração do Sistema Multiagente (Simulação Local)

A implementação completa do sistema multiagente utiliza o **OpenAI Agents SDK** com o endpoint compatível do Google Gemini e está disponível no arquivo `cardioia_agents.py` do repositório GitHub.

Abaixo demonstramos a lógica das tools utilizadas pelos agentes, que são as mesmas registradas com `@function_tool` no sistema real:

- **`predict_risk`**: Consulta o modelo ML treinado para gerar o score de risco.
- **`get_protocols`**: Consulta a base de protocolos médicos simulados.

No sistema real (`cardioia_agents.py`), estas tools são integradas a agentes (`Agent()`) que se comunicam via **handoffs** e o resultado final é validado com **Pydantic**.

In [ ]:
# Demonstração das tools do sistema multiagente

# Tool: predict_risk - utilizada pelo Agente Analista de Risco
def predict_risk(idade, frequencia_cardiaca, spo2, carga_sistema, disponibilidade_recursos, historico_cardiaco):
    """Consulta o modelo preditivo para gerar o score de risco."""
    input_data = pd.DataFrame([{
        'idade': idade,
        'frequencia_cardiaca': frequencia_cardiaca,
        'spo2': spo2,
        'carga_sistema': carga_sistema,
        'disponibilidade_recursos': disponibilidade_recursos,
        'historico_cardiaco': historico_cardiaco
    }])
    prob = model.predict_proba(input_data)[0][1]
    classification = "Alto Risco" if prob > 0.6 else "Baixo Risco"
    return {
        "probabilidade": f"{prob:.2%}",
        "classificacao": classification
    }

# Tool: get_protocols - utilizada pelo Agente Especialista em Protocolos
def get_protocols(classificacao_risco):
    """Consulta a base de protocolos médicos simulados."""
    protocolos = {
        "Alto Risco": [
            "Monitoramento contínuo em UTI.",
            "Administração imediata de antiagregantes plaquetários.",
            "Acionamento da equipe de hemodinâmica."
        ],
        "Baixo Risco": [
            "Observação em enfermaria.",
            "Realização de ECG seriado.",
            "Avaliação cardiológica ambulatorial."
        ]
    }
    return protocolos.get(classificacao_risco, ["Protocolo não encontrado."])

# Simulação do fluxo do sistema multiagente
print("=" * 60)
print("   SIMULAÇÃO DO FLUXO MULTIAGENTE")
print("=" * 60)

# Dados do novo paciente
patient_data = {
    'idade': 68,
    'frequencia_cardiaca': 95,
    'spo2': 92,
    'carga_sistema': 0.7,
    'disponibilidade_recursos': 0.5,
    'historico_cardiaco': 1
}

print(f"\n[Entrada] Dados do paciente: {patient_data}")

# Passo 1: Agente Analista de Risco chama predict_risk
print("\n[Handoff] Orquestrador → Agente Analista de Risco")
risk_result = predict_risk(**patient_data)
print(f"[Analista de Risco] Tool predict_risk → {risk_result}")

# Passo 2: Agente Especialista em Protocolos chama get_protocols
print(f"\n[Handoff] Orquestrador → Agente Especialista em Protocolos")
protocols = get_protocols(risk_result['classificacao'])
print(f"[Especialista em Protocolos] Tool get_protocols → {protocols}")

# Passo 3: Resposta final estruturada
print(f"\n{'=' * 60}")
print("   RELATÓRIO FINAL DA CARDIOIA")
print(f"{'=' * 60}")
print(f"- Probabilidade prevista: {risk_result['probabilidade']}")
print(f"- Classificação de risco: {risk_result['classificacao']}")
print(f"- Protocolos sugeridos:")
for i, p in enumerate(protocols, 1):
    print(f"  {i}. {p}")
print("\nNota: No sistema real (cardioia_agents.py), a resposta é")
print("gerada pelo LLM via OpenAI Agents SDK com validação Pydantic.")

## 5. Entregáveis

Este notebook serve como um dos entregáveis da **Parte 1**, demonstrando a geração de dados, treinamento do modelo, avaliação e simulação. Os demais entregáveis incluem:

*   **Relatório Técnico (Parte 1):** Detalhes sobre o modelo preditivo.
*   **Documento de Arquitetura (Parte 2):** Descrição do sistema multiagente com OpenAI Agents SDK.
*   **Código Completo:** Disponível em repositório GitHub (`cardioia_agents.py`).
*   **Vídeo Demonstrativo:** Link no README do GitHub.
